# Products - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.olist_products"
target_table = f"{catalog}.silver.olist_products"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(bronze_df.limit(10))

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200,38,5,11,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350,70,24,44,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900,40,8,40,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400,27,13,17,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600,17,10,12,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products


In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

Number of rows: 32951
Number of columns: 16


In [0]:
for column, dtype in bronze_df.dtypes[:9]:
    print(column)
    print("Null count:", bronze_df.filter(col(column).isNull()).count())
    print("Distinct count:", bronze_df.filter(col(column).isNotNull()).select(column).distinct().count())
    if dtype == "string":
        print("Extra whitespace row count:",
            (
                bronze_df.withColumn(f"{column}_trimmed", trim(col(column)))
                .filter(col(column) != col(f"{column}_trimmed"))
                .count()
            )
        )
    print("-"*20)

product_id
Null count: 0
Distinct count: 32951
Extra whitespace row count: 0
--------------------
product_category_name
Null count: 610
Distinct count: 73
Extra whitespace row count: 0
--------------------
product_name_lenght
Null count: 610
Distinct count: 66
--------------------
product_description_lenght
Null count: 610
Distinct count: 2960
--------------------
product_photos_qty
Null count: 610
Distinct count: 19
--------------------
product_weight_g
Null count: 2
Distinct count: 2204
--------------------
product_length_cm
Null count: 2
Distinct count: 99
--------------------
product_height_cm
Null count: 2
Distinct count: 102
--------------------
product_width_cm
Null count: 2
Distinct count: 95
--------------------


- product_id does not contain any null value and its distinct count is equal to the number of rows in the table. Hence it is a valid key.
- No extra whitespace was found in the string columns product_id and product_category_name.

In [0]:
missing_product_description_df = bronze_df.filter(
    col("product_category_name").isNull()
    & col("product_name_lenght").isNull()
    & col("product_description_lenght").isNull()
    & col("product_photos_qty").isNull()
)

print(
    "Products missing category, name length, description length, and photo count:",
    missing_product_description_df.count()
)

display(
    missing_product_description_df
    .select("product_id")
    .limit(10)
)

Products missing category, name length, description length, and photo count: 610


product_id
a41e356c76fab66334f36de622ecbd3a
d8dee61c2034d6d075997acef1870e9b
56139431d72cd51f19eb9f7dae4d1617
46b48281eb6d663ced748f324108c733
5fb61f482620cb672f5e586bb132eae9
e10758160da97891c2fdcbc35f0f031d
39e3b9b12cd0bf8ee681bbc1c130feb5
794de06c32a626a5692ff50e4985d36f
7af3e2da474486a3519b0cba9dea8ad9
629beb8e7317703dcc5f35b5463fd20e


In [0]:
display(
    bronze_df.filter(col("product_weight_g").isNull() & col("product_length_cm").isNull() & col("product_height_cm").isNull() & col("product_width_cm").isNull()).groupBy("product_id").count()
)

product_id,count
09ff539a621711667c43eba6a3bd8466,1
5eb564652db742ff8f28759cd8d2652a,1


- The same 610 products have null values in product_category_name, product_name_lenght, product_description_lenght, and product_photos_qty.
- The same 2 products have null values in product_weight_g, product_length_cm, product_height_cm, and product_width_cm.
- These products are retained because they have valid, unique product_id values and remain usable for analyses that do not require the missing attributes.

In [0]:
rescued_row_count = (
    bronze_df
    .filter(col("_rescued_data").isNotNull())
    .filter(trim(col("_rescued_data")) != "")
    .count()
)

print("Number of rescued rows:", rescued_row_count)

Number of rescued rows: 0


There are no rescued data.

## Transform to Silver

In [0]:
translation_table = (f"{catalog}.silver.olist_product_category_translation")

translation_df = spark.table(translation_table)

silver_df = (
    bronze_df.alias("p")
    .join(
        translation_df.alias("t"),
        col("p.product_category_name")
        == col("t.product_category_name_portuguese"),
        "left"
    )
    .select(
        col("p.product_id"),
        col("p.product_category_name").alias("product_category_name_portuguese"),
        col("t.product_category_name_english"),
        col("p.product_name_lenght").alias("product_name_length"),
        col("p.product_description_lenght").alias("product_description_length"),
        col("p.product_photos_qty").alias("product_photo_count"),
        col("p.product_weight_g"),
        col("p.product_length_cm"),
        col("p.product_height_cm"),
        col("p.product_width_cm"),
        col("p._rescued_data"),
        col("p.source_file_path"),
        col("p.source_file_modification_time"),
        col("p.ingestion_timestamp"),
        col("p.ingestion_run_id"),
        col("p.source_system"),
        col("p.source_dataset")
    )
)

In [0]:
unmatched_translation_df = silver_df.filter(
    col("product_category_name_portuguese").isNotNull()
    & col("product_category_name_english").isNull()
)

print("Products with a category but no English translation:", unmatched_translation_df.count())

display(
    unmatched_translation_df
    .select("product_category_name_portuguese")
    .distinct()
)

Products with a category but no English translation: 13


product_category_name_portuguese
pc_gamer
portateis_cozinha_e_preparadores_de_alimentos


- 13 products under two Portuguese categories have no matching English translation.
- These products are retained with a null English category.

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name_portuguese: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- product_name_length: integer (nullable = true)
 |-- product_description_length: integer (nullable = true)
 |-- product_photo_count: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(silver_table_df.limit(10))

product_id,product_category_name_portuguese,product_category_name_english,product_name_length,product_description_length,product_photo_count,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,40,287,1,225,16,10,14,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
3aa071139cb16b67ca9e5dea641aaa2f,artes,art,44,276,1,1000,30,18,20,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,46,250,1,154,18,9,15,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
cef67bcfe19066a932b7673e239eb23d,bebes,baby,27,261,1,371,26,4,26,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares,37,402,4,625,20,17,13,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,musical_instruments,60,745,1,200,38,5,11,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
732bd381ad09e530fe0a5f457d81becb,cool_stuff,cool_stuff,56,1272,4,18350,70,24,44,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,furniture_decor,56,184,2,900,40,8,40,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
37cc742be07708b53a98702e77a21a02,eletrodomesticos,home_appliances,57,163,1,400,27,13,17,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
8c92109888e8cdf9d66dc7e463025574,brinquedos,toys,36,1156,1,600,17,10,12,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products


In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())

Bronze row count: 32951
Silver row count: 32951
